Imports

In [ ]:
# Install optuna if not already installed
# !pip install optuna

import wandb
from xgboost import XGBRegressor
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna

DATA_FILE = "paranal_specific_dataset.csv"

FEATURE_ORDER = [
    "airmass",
    "pressure",
    "temp_2m", "temp_30m", "temp_ground",
    "temp_gradient_ground_30m",
    "dew_point_depression_2m",
    "wind_speed_10m", "wind_speed_30m",
    "wind_shear_10_30",
    "wind_dir_sin", "wind_dir_cos",
    "wind_w_std",
    "rain_intensity",
    "ir_temperature", "pwv",

    "temp_2m_roll3h_mean",
    "temp_2m_trend_3h",
    "wind_speed_10m_roll3h_mean",
    "wind_speed_10m_trend_3h",
    "wind_speed_30m_roll3h_mean",
    "wind_speed_30m_trend_3h",

    "pressure_trend_3h",
    "wind_w_std_roll3h_mean",
]
TARGET = "seeing"

Data

In [ ]:
try:
    raw_dataset = pd.read_csv(DATA_FILE, encoding="utf-8")
    print(f"data is loaded: {len(raw_dataset)} rows")
except FileNotFoundError:
    raise SystemExit(
        f"{DATA_FILE} is not found"
    )

dataset = raw_dataset.copy()

data is loaded: 36709 rows


Wandb

In [ ]:
wandb.init(
    project="seeing-prediction",
    name="XGboost-v4"
)

Add Feature

In [ ]:
wind_rad = np.deg2rad(dataset["wind_dir_10m"])
dataset["wind_dir_sin"] = np.sin(wind_rad)
dataset["wind_dir_cos"] = np.cos(wind_rad)

dataset = dataset.sort_values("datetime").reset_index(drop=True)

for col in ["temp_2m", "wind_speed_10m", "wind_speed_30m"]:
    dataset[f"{col}_roll3h_mean"] = dataset[col].rolling(3, min_periods=1).mean()
    dataset[f"{col}_trend_3h"] = dataset[col] - dataset[col].shift(3)

dataset["pressure_trend_3h"] = dataset["pressure"] - dataset["pressure"].shift(3)
dataset["wind_w_std_roll3h_mean"] = dataset["wind_w_std"].rolling(3, min_periods=1).mean()

Read Data

In [ ]:
valid_subset_cols = [col for col in (FEATURE_ORDER + [TARGET]) if col in dataset.columns]
dataset = dataset.dropna(subset=valid_subset_cols)
print(f"after clean data: {len(dataset)} rows")

dataset["log_seeing"] = np.log1p(dataset[TARGET])
LOG_TARGET = "log_seeing"

after clean data: 36706 rows


Split the data into training, test and cv data sets

In [ ]:
dataset = dataset.sort_values("datetime").reset_index(drop=True)
dataset["week_block"] = pd.to_datetime(dataset["datetime"], utc=True).dt.to_period("W").astype(str)

blocks_in_order = dataset["week_block"].drop_duplicates().tolist()
n_blocks = len(blocks_in_order)

n_test_blocks = max(1, round(n_blocks * 0.15))
test_blocks = set(blocks_in_order[-n_test_blocks:])
remaining_blocks = blocks_in_order[:-n_test_blocks]

rng_blocks = np.random.default_rng(0)
remaining_blocks = list(remaining_blocks)
rng_blocks.shuffle(remaining_blocks)
n_cv_blocks = max(1, round(len(remaining_blocks) * (0.15 / 0.85)))
cv_blocks = set(remaining_blocks[:n_cv_blocks])
train_blocks = set(remaining_blocks[n_cv_blocks:])

train_dataset = dataset[dataset["week_block"].isin(train_blocks)]
cv_dataset = dataset[dataset["week_block"].isin(cv_blocks)]
test_dataset = dataset[dataset["week_block"].isin(test_blocks)]

bins = [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 100]
print(pd.cut(test_dataset[TARGET], bins=bins).value_counts().sort_index())

Split features from labels

In [ ]:
x_train = train_dataset[FEATURE_ORDER].select_dtypes(include=np.number).copy()
x_cv = cv_dataset[FEATURE_ORDER].select_dtypes(include=np.number).copy()
x_test = test_dataset[FEATURE_ORDER].select_dtypes(include=np.number).copy()

y_train = train_dataset[LOG_TARGET].copy()
y_cv = cv_dataset[LOG_TARGET].copy()
y_test_real = test_dataset[TARGET].copy()

wandb.config.update({
    "model": "XGboost",
    "n_features": len(FEATURE_ORDER),
    "train_samples": len(x_train),
    "test_samples": len(x_test)
})

XGboost

In [ ]:
n_estimators=1577
learning_rate=0.012573869938580187
max_depth=5
subsample=0.651710104051176
colsample_bytree=0.9376970674069666
min_child_weight=4
reg_alpha=2.106866853816057
reg_lambda=0.009730146712099774
gamma=0.005263852817833482

model = XGBRegressor(
    n_estimators=n_estimators,
    learning_rate=learning_rate,
    max_depth=max_depth,
    subsample=subsample,
    colsample_bytree=colsample_bytree,
    min_child_weight=min_child_weight,
    reg_alpha=reg_alpha,
    reg_lambda=reg_lambda,
    gamma=gamma,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50
)

model.fit(x_train, y_train, eval_set=[(x_cv, y_cv)], verbose=False)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9376970674069666, device=None,
             early_stopping_rounds=50, enable_categorical=True,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=0.005263852817833482, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.012573869938580187,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=4, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1577, n_jobs=-1,
             num_parallel_tree=None, ...)

predict

In [ ]:
pred_log = model.predict(x_test)

pred = np.expm1(pred_log)

print(pred[:5])
print(y_test_real[:5])

[0.6762998  0.71287704 1.0521035  1.0283253  0.93010825]
30918    0.521390
30919    0.538841
30920    0.734489
30921    0.931722
30922    1.359432
Name: seeing, dtype: float64


Cheking Model

In [ ]:
mae = mean_absolute_error(
  y_test_real,
  pred
)

mse = mean_squared_error(
  y_test_real,
  pred
)

rmse = np.sqrt(mse)

r2 = r2_score(
  y_test_real,
  pred
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 0.15953474389199288
RMSE: 0.2253015299241272
R2: 0.6963170544638513


In [ ]:
baseline_pred = np.expm1(model.predict(x_test).flatten())
baseline_mae = np.mean(np.abs(baseline_pred - y_test_real))

importances = {}
rng = np.random.default_rng(0)
for col in x_train.columns:
    shuffled = x_test.copy()
    shuffled[col] = rng.permutation(shuffled[col].values)
    shuffled_pred = np.expm1(model.predict(shuffled).flatten())
    shuffled_mae = np.mean(np.abs(shuffled_pred - y_test_real))
    importances[col] = shuffled_mae - baseline_mae

importance_series = pd.Series(importances).sort_values(ascending=False)
print(importance_series)

wind_dir_sin                  0.039947
temp_2m                       0.027222
wind_speed_30m                0.024810
wind_speed_10m_roll3h_mean    0.022531
wind_speed_10m                0.013376
wind_dir_cos                  0.013102
wind_w_std                    0.009815
temp_30m                      0.007631
temp_2m_trend_3h              0.005061
wind_shear_10_30              0.004765
wind_w_std_roll3h_mean        0.003566
temp_2m_roll3h_mean           0.001614
pwv                           0.001393
pressure                      0.001057
temp_gradient_ground_30m      0.001047
temp_ground                   0.001042
rain_intensity                0.000924
wind_speed_30m_roll3h_mean    0.000840
wind_speed_10m_trend_3h       0.000768
pressure_trend_3h             0.000718
ir_temperature                0.000560
wind_speed_30m_trend_3h       0.000478
dew_point_depression_2m       0.000290
airmass                       0.000282
dtype: float64


In [ ]:
wandb.log({
    "n_estimators": n_estimators,
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "learning_rate": learning_rate,
    "max_depth": max_depth
})


wandb.finish()

MAE,▁
R2,▁
RMSE,▁
learning_rate,▁
max_depth,▁
n_estimators,▁
MAE,0.15953
R2,0.69632
RMSE,0.2253
learning_rate,0.01257
max_depth,5


Save Model

In [ ]:
model.save_model("xgb_model.json")